# 정적 웹페이지 데이터 수집

## 실습 주제
Books to Scrape 웹사이트에 HTTP 요청을 보내고, 서버가 반환한 원본 HTML을 확인하고 저장
- https://books.toscrape.com/

In [2]:
from pathlib import Path
from datetime import datetime

import requests


## ===========================================================
## 1. 수집 설정
## ===========================================================

# TARGET_URL = 'https://books.toscrape.com/'

## 의도적으로 오류 발생시키기 : 404
TARGET_URL = 'https://books.toscrape.com/not-found-page.html'

CONNECT_TIMEOUT  = 5
READ_TIMEOUT = 30

HEADERS = {'User-Agent': 'EducationalDataCollector/1.0'}


## ===========================================================
## 2. 저장 폴더 생성
## ===========================================================

PROJECT_DIR = Path('D:/AI/data_analytics/crawling/01-data-collection-pipeline')
raw_html_dir = PROJECT_DIR / 'data' / 'raw' / 'html'

raw_html_dir.mkdir(
    parents=True, ## 부모 폴더가 없다면 생성  
    exist_ok=True, ## 이미 존재하는 경로라면 Error 발생 안 함
)


## ===========================================================
## 3. 웹 페이지 요청
## ===========================================================
try:
    response = requests.get(
        TARGET_URL,
        headers=HEADERS,
        timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
    )

    ## 응답 상태 검사
    ## 서버가 반환한 HTTP 상태코드가 오류인지 검사하고, 오류라면 예외 발생
    ## 400번대, 500번대 상태코드는 HTTPError 발생
    print(f'응답 상태코드 : {response.status_code}')
    response.raise_for_status()

## ===========================================================
## 4. 요청 오류 처리
## ===========================================================
except requests.exceptions.HTTPError as error:
    print('HTTP 응답 오류가 발생했습니다.')
    print(f'오류 내용 : {error}')

except requests.exceptions.RequestException as error:
    print('웹페이지 요청 중 오류가 발생했습니다.')
    print(f'오류 내용 : {error}')

## ===========================================================
## 5. 요청 성공 처리
## ===========================================================
else:
    collected_at = datetime.now()
    timestamp = collected_at.strftime('%Y%m%d_%H%M%S')

    raw_file = raw_html_dir / f'books_home_{timestamp}.html'
    raw_file.write_bytes(response.content)

    print('웹페이지 수집을 완료했습니다.')
    print('=' * 60)
    print(f'요청 URL : {TARGET_URL}')
    print(f'최종 URL : {response.url}')
    print(f'상태 코드 : {response.status_code}')
    print(f'Content-Type : {response.headers.get('Content-Type')}')
    print(f'응답 인코딩 : {response.encoding}')
    print(f'본문 기준 추정 인코딩 : {response.apparent_encoding}')
    print(f'응답 크기 : {len(response.content):,} bytes')
    print(f'수집 시각 : {collected_at: %Y-%m-%d %H:%M:%S}')    
    print(f'원본 HTML 저장 경로 : {raw_file}')    

응답 상태코드 : 404
HTTP 응답 오류가 발생했습니다.
오류 내용 : 404 Client Error: Not Found for url: https://books.toscrape.com/not-found-page.html
